# Single-Wavelength CG Optimization

Single-wavelength hologram optimization using a slmsuite wavefront-calibration input beam.

## 1. Imports and Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from initial_holograms import curvature_hologram
from loss_functions import build_circular_mask
from optimizer import CGOptimizer
from optical_planes import CameraPlane, SLMPlane
from propagator import Propagator
from target_profiles import apply_psf_smoothing, build_rectangle_target

# Native SLM and compute grid.
full_slm_shape = (1080, 1920)
superpixel_size = 4
slm_shape = (full_slm_shape[0] // superpixel_size, full_slm_shape[1] // superpixel_size)
padding_factor = 2
camera_shape = (slm_shape[0] * padding_factor, slm_shape[1] * padding_factor)

# Optical parameters.
wavelength_nm = 420.0
slm_pixel_pitch_um = 8.0
camera_pixel_pitch_um = 3.45
focal_length_mm = 200.0

# slmsuite wavefront-calibration input beam.
slmsuite_calibration_h5_path = "10806-SLM-wavefront_superpixel-calibration_00036.h5"
slmsuite_camera_shape = full_slm_shape
slmsuite_wavefront_r2_threshold = 0.5
slmsuite_remove_background = True
slmsuite_apply_calibration = True
slmsuite_amplitude_key = "amplitude"
slmsuite_phase_key = "phase"

# Target and mask parameters in focal-plane physical units.
rectangle_width_x_um = 300.0
rectangle_width_y_um = 100.0
psf_sigma_x_um = 10.0
psf_sigma_y_um = 10.0
mask_radius_margin_um = 100.0

# Initial shared hologram parameters.
initial_phase_linear_tilt = 0.0
initial_phase_astigmatism_weight = 0.0
initial_phase_quadratic_curvature = 3.6e-3
initial_phase_linear_angle_rad = np.pi / 4.0
initial_phase_conical_weight = 0.0

# CG parameters.
optimizer_maxiter = 100
loss_scale = 1e12
optimize_phase = True


## 2. Build Optical System and Target

In [ ]:
def downsample_by_superpixel(data, superpixel_size):
    data_array = np.asarray(data, dtype=float)
    if data_array.ndim != 2:
        raise ValueError(f"data must be 2D, got shape {data_array.shape}.")
    if data_array.shape[0] % superpixel_size != 0 or data_array.shape[1] % superpixel_size != 0:
        raise ValueError(f"data shape must be divisible by superpixel_size, got data_shape={data_array.shape}, superpixel_size={superpixel_size}.")
    ny = data_array.shape[0] // superpixel_size
    nx = data_array.shape[1] // superpixel_size
    return data_array.reshape(ny, superpixel_size, nx, superpixel_size).mean(axis=(1, 3))


def downsample_phase_by_superpixel(phase, superpixel_size):
    phase_array = np.asarray(phase, dtype=float)
    if phase_array.ndim != 2:
        raise ValueError(f"phase must be 2D, got shape {phase_array.shape}.")
    real_part = downsample_by_superpixel(np.cos(phase_array), superpixel_size)
    imag_part = downsample_by_superpixel(np.sin(phase_array), superpixel_size)
    return np.angle(real_part + 1j * imag_part)


def require_calibration_array(calibration_results, key, expected_shape):
    if key not in calibration_results:
        available_keys = sorted(str(item) for item in calibration_results.keys())
        raise KeyError(f"Calibration result key {key!r} was not found. Available keys: {available_keys}.")
    array = np.asarray(calibration_results[key], dtype=float)
    if array.shape != expected_shape:
        raise ValueError(f"Calibration result {key!r} shape must be {expected_shape}, got {array.shape}.")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"Calibration result {key!r} contains NaN or infinite values.")
    return array


def load_slmsuite_input_beam_and_phase(calibration_h5_path, full_slm_shape, superpixel_size, slm_pixel_pitch_um, wavelength_nm, camera_shape, r2_threshold, remove_background, apply_calibration, amplitude_key, phase_key):
    from slmsuite.hardware.cameras.simulated import SimulatedCamera
    from slmsuite.hardware.cameraslms import FourierSLM
    from slmsuite.hardware.slms.simulated import SimulatedSLM

    slm_size_xy = (int(full_slm_shape[1]), int(full_slm_shape[0]))
    camera_size_xy = (int(camera_shape[1]), int(camera_shape[0]))
    slm = SimulatedSLM(slm_size_xy, pitch_um=(slm_pixel_pitch_um, slm_pixel_pitch_um), wav_um=wavelength_nm / 1000.0)
    camera = SimulatedCamera(slm, resolution=camera_size_xy)
    fourier_slm = FourierSLM(camera, slm)
    fourier_slm.load_calibration("wavefront_superpixel", file_path=calibration_h5_path)
    calibration_results = fourier_slm.wavefront_calibration_superpixel_process(
        plot=False,
        r2_threshold=r2_threshold,
        remove_background=remove_background,
        apply=apply_calibration,
    )
    amplitude_full = require_calibration_array(calibration_results, amplitude_key, full_slm_shape)
    phase_full = require_calibration_array(calibration_results, phase_key, full_slm_shape)
    amplitude = downsample_by_superpixel(np.clip(amplitude_full, 0.0, None), superpixel_size)
    phase = downsample_phase_by_superpixel(phase_full, superpixel_size)
    if np.max(amplitude) <= 0:
        raise ValueError("Loaded slmsuite amplitude contains no positive values after downsampling.")
    return amplitude / np.max(amplitude), np.mod(phase, 2.0 * np.pi), calibration_results


input_beam_amplitude, input_beam_phase, slmsuite_calibration_results = load_slmsuite_input_beam_and_phase(
    calibration_h5_path=slmsuite_calibration_h5_path,
    full_slm_shape=full_slm_shape,
    superpixel_size=superpixel_size,
    slm_pixel_pitch_um=slm_pixel_pitch_um,
    wavelength_nm=wavelength_nm,
    camera_shape=slmsuite_camera_shape,
    r2_threshold=slmsuite_wavefront_r2_threshold,
    remove_background=slmsuite_remove_background,
    apply_calibration=slmsuite_apply_calibration,
    amplitude_key=slmsuite_amplitude_key,
    phase_key=slmsuite_phase_key,
)
initial_hologram = curvature_hologram(
    shape=slm_shape,
    linear_tilt=initial_phase_linear_tilt,
    astigmatism_weight=initial_phase_astigmatism_weight,
    quadratic_curvature=initial_phase_quadratic_curvature,
    linear_angle_rad=initial_phase_linear_angle_rad,
    conical_weight=initial_phase_conical_weight,
    center=(slm_shape[0] / 2.0, slm_shape[1] / 2.0),
)

slm = SLMPlane(slm_shape, wavelength_nm, slm_pixel_pitch_um, superpixel_size, input_beam_amplitude, input_beam_phase, initial_hologram)
camera = CameraPlane(camera_shape, wavelength_nm, (1.0, 1.0), camera_pixel_pitch_um, np.zeros(camera_shape), np.zeros(camera_shape))
propagator = Propagator(slm, camera, focal_length_mm, padding_factor)

ideal_target = build_rectangle_target(camera_shape, propagator.camera_plane.x_axis_um, propagator.camera_plane.y_axis_um, rectangle_width_x_um, rectangle_width_y_um)
target_amplitude = apply_psf_smoothing(ideal_target, psf_sigma_x_um, psf_sigma_y_um, propagator.camera_plane.scale_um)
target_phase = np.zeros_like(target_amplitude)
mask_center_x_um = 0.5 * (propagator.camera_plane.x_axis_um[0] + propagator.camera_plane.x_axis_um[-1])
mask_center_y_um = 0.5 * (propagator.camera_plane.y_axis_um[0] + propagator.camera_plane.y_axis_um[-1])
mask_radius_um = max(rectangle_width_x_um, rectangle_width_y_um) / 2.0 + mask_radius_margin_um
target_mask = build_circular_mask(camera_shape, propagator.camera_plane.x_axis_um, propagator.camera_plane.y_axis_um, mask_center_x_um, mask_center_y_um, mask_radius_um)
target_plane = CameraPlane(camera_shape, wavelength_nm, propagator.camera_plane.scale_um, camera_pixel_pitch_um, target_amplitude, target_phase)
optimizer = CGOptimizer(propagator, target_plane, target_mask)
optimizer.set_initial_hologram_array(initial_hologram)

print("SLM shape:", slm.shape)
print("Camera/optimization shape:", camera_shape)
print("Input beam shape:", input_beam_amplitude.shape)
print("Target active pixels:", int(np.sum(target_mask > 0)))

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), constrained_layout=True)
axes[0].imshow(input_beam_amplitude**2, origin="lower", cmap="magma", aspect="equal")
axes[0].set_title("Input intensity")
axes[1].imshow(input_beam_phase, origin="lower", cmap="twilight", aspect="equal")
axes[1].set_title("Input phase")
axes[2].imshow(target_amplitude**2 * target_mask, origin="lower", cmap="inferno", aspect="equal")
axes[2].set_title("Target and mask")
plt.show()


## 3. Optimize

In [ ]:
optimizer.optimize(
    maxiter=optimizer_maxiter,
    loss_scale=loss_scale,
    optimize_phase=optimize_phase,
)

print("Optimization finished")
print("Loss evaluations:", len(optimizer.loss_history))
print("Accepted CG iterations:", len(optimizer.iteration_loss_history))


## 4. Show Result

In [ ]:
result = optimizer.get_result_summary()
print("Efficiency:", result.efficiency)
print("Fidelity:", result.fidelity)
print("RMS error:", result.rms_error)
print("Phase error:", result.phase_error)
print("Optimization time (s):", result.optimization_time_sec)

optimizer.plot_loss_history()
plt.show()

optimizer.plot_result_summary()
plt.show()
